# 3.5 텐서 요소 타입
Tensor에 어떤 타입의 값을 저장할 수 있을까?
- 파이썬에서 숫자는 객체다.
  - 언박싱: 소량 저장은 O, 대량 저장은 비효율적
- 파이썬에서 리스트는 연속된 객체의 컬렉션이다.
  - 벡터 내적 연산과 합을 수행 X, 파이썬 리스트는 단일 차원
- 파이썬 인터프리터는 최적화를 거치는 컴파일된 코드보다 느리다.
  - 다량의 숫자 데이터 모음에 대한 수학적 연산 수행이 느리다.

넘파이, 파이토치 텐서와 같이 전용 데이터 구조를 만든 후 연산의 효율성을 높이도록 구현하고 고차원 API로 편리성을 더한다.

성능 최적화를 위해 텐선 내의 모든 객체는 같은 타입의 숫자여야 하고, 파이토치는 실행 중에 계속 추적이 가능해야 한다.

## 3.5.1 dtype으로 숫자 타입 저장하기
tensor, zeros, ones 같은 텐서 생성자 실행 시 넘겨주는 dtype 인자로 텐서 내부에 들어갈 데이터 타입을 지정할 수 있다.
- torch.float
- torch.double
- torch.half
- torch.int8

등등

## 3.5.2 모든 경우에 사용하는 dtype
신경망 연산은 대부분 32비트 부동소수점 연산.

정확도를 희생해 정밀도를 반으로 떨어뜨려 신경망이 차지하는 공간을 줄이는 방식 가능

텐서는 다른 텐서에 대한 인덱스로 사용할 수 있다.

텐서를 만들 때 torch.tensor([2, 2])처럼 인자로 정수값을 주면 64비트 정수 텐서를 기본으로 만든다.

## 3.5.3 텐서의 dtype 속성 관리
숫자 타입이 올바르게 지정된 텐서를 하나 할당할 때에는 생성자에 dtype 인자를 정확하게 전달해야 한다.

In [2]:
import torch

double_points = torch.ones(10, 2, dtype=torch.double)
short_points = torch.tensor([[1, 2], [3, 4]], dtype=torch.short)

# 어떤 텐서가 가진 dtype을 알고 싶다면
short_points.dtype

torch.int16

In [ ]:
# 텐서 생성 함수가 반환하는 텐서의 타입을 대응하는 캐스팅 메소드 사용(올바른 타입 변환)
# 캐스팅 메소드란 텐서의 `dtype` 속성을 직접적으로 변경하는 역할(.double(), .short())
double_points = torch.zeros(10, 2).double()
short_points = torch.ones(10, 2).short()

In [ ]:
# 혹은 to 메소드를 사용
double_points = torch.zeros(10, 2).to(torch.double)
short_points = torch.ones(10, 2).to(dtype=torch.short)
# to 메소드는 변환이 필요한 경우에만
# float 같은 dtype 이름을 사용한 캐스팅보다 to 메소드는 타입 외에도 추가적인 인자 지정 가능

In [ ]:
# 여러 타입을 가진 입력들이 연산을 거치며 서로 섞일 때 자동으로 제일 큰 타입 생성
points_64 = torch.rand(5, dtype=torch.double) # rand는 텐서 요소를 0과 1 사이 임의의 수로 초기화
points_short = points_64.to(torch.short)
points_64 * points_short # 결과는 double 타입

tensor([0., 0., 0., 0., 0.], dtype=torch.float64)

## 3.6 텐서 API
텐서끼리의 연산 대부분은 torch 모듈에 있고 대부분이 텐서 객체에 대해 메소드처럼 호출할 수 있다.(transpose 함수도 torch 모듈로 호출)


In [ ]:
a = torch.ones(3, 2)
a_t = torch.transpose(a, 0, 1)

a.shape, a_t.shape

(torch.Size([3, 2]), torch.Size([2, 3]))

In [ ]:
# 텐서 메소드
a = torch.ones(3, 2)
a_t = a.transpose(0, 1)

a.shape, a_t.shape

(torch.Size([3, 2]), torch.Size([2, 3]))

파이토치 온라인 문서(http://pytorch.org/docs)에 완벽한 텐서 연산 정리(API 함수)

## 3.7 텐서를 저장소 관점에서 머릿속에 그려보기
텐서의 내부 값은 torch.Storage 인스턴스로 관리하며 연속적인 메모리 조각으로 할당된 상태다. 저장 공간은 숫자 데이터를 가진 1차원 배열이다.

- 64비트 공간의 int64 타입 숫자들이 연속해서 들어있는 메모리 블럭
- Tensor 객체: 저장 공간을 나타내는 Storage 객체에 대한 뷰 역할 담당
  - 오프셋을 사용해 공간의 임의 위치에 접근하거나 특정 차원의 크기를 단위로 접근

### 3.7.1 저장 공간 인덱싱
저장 공간 인덱싱은 텐서를 위한 저장공간은 .storage 프로퍼티로 접근

In [ ]:
points = torch.tensor(([4.0, 1.0], [5.0, 3.0], [2.0, 1.0]))
points.storage()

<ipython-input-10-4334df3db23b>:2: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  points.storage()


 4.0
 1.0
 5.0
 3.0
 2.0
 1.0
[torch.storage.TypedStorage(dtype=torch.float32, device=cpu) of size 6]

이 텐서는 3개의 행과 2개의 열로 이뤄져있지만 실제 크기가 6인 배열 공간이다.
텐서는 주어진 차원 쌍이 실제로 어느 공간에 해당하는지 알 뿐이다.
텐서를 거치지 않고 저장 공간을 다음과 같이 직접 접근 가능하다.

In [ ]:
points_storage = points.storage()
points_storage[0]

4.0

In [ ]:
points.storage()[1]

1.0

2차원 텐서라고 해서 저장 공간을 두 개의 인덱스로 2차원처럼 접근하진 않는다. 차원에 무관하게 **실제 공간 레이아웃은 1차원**이다.

In [ ]:
# 저장 공간 값을 바꾸면 참조하고 있는 텐서에서의 내용 변경
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points_storage = points.storage()
points_storage[0] = 2.0  # 첫번째 인덱스 값 바꾸기
points

tensor([[2., 1.],
        [5., 3.],
        [2., 1.]])

### 3.7.2 저장된 값을 수정하기: 텐서 내부 연산
- zero_ : 밑줄로 끝나는데 연산의 결과로 새 텐서가 넘어오는 대신 기존 텐서의 내용이 바뀐다

밑줄로 끝나지 않는 모든 메소드들은 원래 텐서는 그대로 두고 새로운 텐서를 만들어 넘김

In [ ]:
a = torch.ones(3, 2)

a.zero_()
a

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])

## 3.8 텐서 메타데이터: 사이즈, 오프셋, 스트라이드
- **사이즈, 오프셋, 스트라이드**: 저장 공간을 인덱스로 접근하기 위해 텐서는 저장 공간에 포함된 몇 가지 명확한 정보에 의존
  - 텐서의 **사이즈**는 텐서의 각 차원 별로 들어가는 요소의 수를 표시한 튜플
  - 저장 공간에 대한 **오프셋**은 텐서의 첫 번째 요소를 가리키는 색인 값과 동일
  - **스트라이드**는 각 차원에서 다음 요소를 가리키고 싶을 때 실제 저장 공간상에서 몇개의 요소를 건너뛰어야 하는지를 알려주는 숫자

### 3.8.1 다른 텐서의 저장 공간에 대한 뷰 만들기
대응하는 인덱스를 제공해서 텐서에 들어 있는 두 번째 포인트를 얻기

In [ ]:
points = torch.tensor(([4.0, 1.0], [5.0, 3.0], [2.0, 1.0]))
second_point = points[1]
second_point.storage_offset()

2

In [ ]:
second_point.size()

torch.Size([2])

두 번째 포인트는 저장 공간에서 첫 포인트의 값 두 개 다음이니깐 오프셋2에 해당하는 위치에 있다. 그리고 두 번째 포인트는 1차원이므로 Size 클래스는 하나의 요소를 가지는 객체이고 값으로는 차원 크기가 들어 있다.

In [ ]:
second_point.shape

torch.Size([2])

스트라이드는 값을 가진 튜플인데, 각 차원에서 인덱스를 하나 증가했을 때 저장 공간상에서 몇 개 요소를 건너뛰어야 하는지를 값으로 가진다. Points 텐서는 (2, 1) 스트라이드를 가진다.

In [ ]:
points.stride()

(2, 1)

2차원 텐서에서 요소 i, j에 접근한다면?

저장 공간상으로 `storage_offset + stride[0] * i + stride[1] * j` 번째 요소

사이즈 값

In [ ]:
second_point = points[1]
second_point.size()

torch.Size([2])

오프셋 값

In [ ]:
second_point.storage_offset()

2

스트라이드 값

In [ ]:
second_point.stride()

(1,)

중요한점: 예상대로 새 텐서가 원래의 points 텐서보다 하나 작은 차원을 가지지만 여전히 동일한 저장 공간을 가리키고 있다는 점.

(새 텐서를 변경하면 원래의 텐서도 변경)

In [ ]:
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
second_point = points[1]
second_point[0] = 10.0
points

tensor([[ 4.,  1.],
        [10.,  3.],
        [ 2.,  1.]])

서브텐서를 새 텐서로 복제(.clone)

In [ ]:
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
second_point = points[1].clone()
second_point[0] = 10.0
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

### 3.8.2 복사 없이 텐서 전치하기
points 텐서에는 행마다 개별 포인트가 들어 있고 열에는 X와 Y좌표를 가진다. 이를 전치하여 열에 포인트가 들어가도록 만들자. 2차원 텐서들에 대해 transpose를 사용해도 되고 다음과 같이 t 함수를 사용해도 된다.

In [9]:
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [12]:
points_t = points.t()
points_t

tensor([[4., 5., 2.],
        [1., 3., 1.]])

In [ ]:
# 두 텐서가 같은 공간을 가리키고 있는지 확인
id(points.storage()) == id(points_t.storage())

True

In [ ]:
# 두 텐서는 차원 정보와 스트라이드만 다름
points.stride()

(2, 1)

In [ ]:
points_t.stride()

(1, 2)

### 3.8.3 더 높은 차원에서의 전치 연산
파이토치의 전치 연산은 행렬에만 국한되지 않고, 다차원 배열에 대해서도 차원 정보와 스트라이드가 바뀔 두 차원을 각각 지정해주면 전치된다.

In [3]:
some_t = torch.ones(3, 4, 5)
transpose_t = some_t.transpose(0, 2)
some_t.shape

torch.Size([3, 4, 5])

In [4]:
transpose_t.shape

torch.Size([5, 4, 3])

In [5]:
some_t.stride()

(20, 5, 1)

In [6]:
transpose_t.stride()

(1, 5, 20)

가장 오른쪽 차원에서 시작해서 증가되는 형태.

저장소에 값이 펼쳐진 텐서(2차원 텐서에서 열을 따라 이동)는 contiguous로 정의.

인접한 텐서는 값 순회 시 띄엄띄엄 참조하지 않기 때문에 데이터 지역성 관점에서 CPU 메모리 접근 효율이 좋다.


### 3.8.4 인접한 텐서
인접한 텐서에 대해서만 동작하는 경우, contiguous가 실제로 하는 일은 없고 성능에 지장을 주지도 않는다.

In [10]:
points.is_contiguous()

True

In [13]:
points_t.is_contiguous()

False

contiguous 메소드를 사용하면 인접하지 않은 텐서를 인접한 텐서로 만들 수도 있다. 텐서 내용은 동일하나, 값의 배치와 스트라이드가 바뀐 텐서가 만들어진다.

In [14]:
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points_t = points.t()
points_t

tensor([[4., 5., 2.],
        [1., 3., 1.]])

In [15]:
points_t.storage()

<ipython-input-15-8a267790edd6>:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  points_t.storage()


 4.0
 1.0
 5.0
 3.0
 2.0
 1.0
[torch.storage.TypedStorage(dtype=torch.float32, device=cpu) of size 6]

In [16]:
points_t.stride()

(1, 2)

In [17]:
points_t_cont = points_t.contiguous()
points_t_cont

tensor([[4., 5., 2.],
        [1., 3., 1.]])

In [18]:
points_t_cont.stride() # 바뀐 텐서

(3, 1)

In [19]:
points_t_cont.storage() # 내용은 동일

 4.0
 5.0
 2.0
 1.0
 3.0
 1.0
[torch.storage.TypedStorage(dtype=torch.float32, device=cpu) of size 6]

항목을 열 단위로 배치하기 위해 새 저장 공간을 재편성한 것에 주목하자. 새로운 배치에 따라 스트라이드도 바뀜